In [69]:
from tool_server.utils.utils import *
from tool_server.utils.prompts import tool_planning_model_prompt_one_tool_call
from tqdm import tqdm
import json, os

input_data = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/refocus/data/refocus_chartqa_v_bar_wbb_selfbar_reformatted.jsonl"

input_data2 = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/refocus/data/refocus_chartqa_v_bar_wbb_reformatted.jsonl"

input_data = process_jsonl(input_data)
input_data2 = process_jsonl(input_data2)
output_path = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/refocus/data/sft_data/refocus_chartqa_v_bar_wbb_selfbar_candidates.json"
output_path2 = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/refocus/data/sft_data/refocus_chartqa_v_bar_wbb_reformatted.json"

refocus_data_dir = "/mnt/petrelfs/songmingyang/songmingyang/data/vl_reasoning/ReFocus_data/train_chartQA_v_bar_wbb/gpt-4o-2024-08-06-mix_orig_build_train_nogold"

tool_name_list = ["OCR","HighlightBox","GroundingDINO","DrawLine","LanguageModel","SegmentRegionAroundPoint","GetSubplotInfo","Crop","Point","DrawShape","MaskBox","GetBarInfo"]


In [70]:
def turn_into_sft_format(input_data):
    sft_data = []
    tool_statistic_dict = {}
    average_tool_call_list = []
    conv_unqualify = 0
    for item in tqdm(input_data):
        conversations = item["response_messages"]
        origin_qid = item["origin_qid"]
        qid = item["qid"]
        current_dir = os.path.join(refocus_data_dir, origin_qid)
        
        jpg_files = [f for f in os.listdir(current_dir) 
                if (f.endswith(".jpg") or f.endswith(".png")) and os.path.isfile(os.path.join(current_dir, f))]
        qid_file_name = f"{origin_qid}.jpg"
        jpg_files.sort(key=lambda x: 0 if x == qid_file_name else 1)
        jpg_files = [os.path.join(current_dir, f) for f in jpg_files]
        
        new_conversations = [{"from":"system","value":tool_planning_model_prompt_one_tool_call}]
        image_cnt = 0
        image_indexes = []
        terminate = False
        current_tool_call_num = 0
        # print(f"Processing conversation: {conversations}")
        if not isinstance(conversations, list):
            conversations = [conversations]
        if len(conversations) <= 2:
            continue
        
        for conv in conversations:
            # print(f"Processing conversation: {conv}")
            if "content" not in conv or "role" not in conv:
                terminate = True
                break
            
            new_role = "gpt" if conv["role"] == "assistant" else "human"
            contents = conv["content"]
            if conv["role"] == "system":
                continue
            new_value = ""
            for content in contents:
                # print(f"Processing content: {content}")
                if isinstance(content, str) or "text" in content:
                    if isinstance(content, str):
                        text_content = f"{content}"
                    elif "text" in content:
                        text_content = f"{content['text']}"
                    text_content = text_content.replace("<image>","")
                    new_value += text_content
                    if "<tool_call>" in text_content:
                        tool_json = text_content.split("<tool_call>")[-1].split("</tool_call>")[0]
                        
                        try:
                            tool_content = json.loads(tool_json)
                            tool_name = tool_content["name"]
                            
                            assert tool_name in tool_name_list, f"Tool name {tool_name} not in predefined list."
                            current_tool_call_num += 1
                            tool_statistic_dict[tool_name] = tool_statistic_dict.get(tool_name, 0) + 1
                            
                        except:
                            conv_unqualify += 1
                            tool_name = "invalid"
                            qualify = 0
                            terminate = True
                            break
                    
                elif "image" in content or "image_url" in content:
                    new_value += f"\n<image>\n"
                    image_cnt += 1
                    image_content = "img_1"
                    if content.get("image_url"):
                        if isinstance(content["image_url"], str):
                            image_content = content["image_url"]
                        else:
                            image_content = content["image_url"].get("url", "img_1")
                    else:
                        image_content = content.get("image", "img_1")
                    
                    image_index = int(image_content.split("_")[-1])-1
                    image_index = 1 if image_index > 1 else image_index
                    image_indexes.append(image_index)
    
                elif "tool_code" in content or "tool_call" in content or "tool_response" in content or "tool_response_from" in content:
                    terminate = True
                    break
                else:
                    terminate = True
                    break
                    # raise ValueError(f"Unknown content type: {content['type']}")
            if terminate:
                break
            new_conversations.append({"from": new_role, "value": new_value})
        if terminate:
            continue
        average_tool_call_list.append(current_tool_call_num)
        
        new_jpg_files = [jpg_files[i] for i in image_indexes]
        assert image_cnt == len(new_jpg_files), f"Image count mismatch for {origin_qid}: {image_cnt} vs {len(jpg_files)}"
        new_item = {
            "qid": qid,
            "conversations": new_conversations,
            "images": new_jpg_files,
        }
        sft_data.append(new_item)
        
    average_tool_call = sum(average_tool_call_list) / len(average_tool_call_list)
    return {
        "sft_data": sft_data,
        "tool_statistic_dict": tool_statistic_dict,
        "average_tool_call": average_tool_call,
    }

In [71]:
data1_turn_res = turn_into_sft_format(input_data)
sft_data, tool_statistic_dict, average_tool_call = data1_turn_res["sft_data"], data1_turn_res["tool_statistic_dict"], data1_turn_res["average_tool_call"]
len(sft_data), tool_statistic_dict, average_tool_call,

100%|██████████| 400/400 [00:01<00:00, 376.85it/s]


(379, {'HighlightBox': 362, 'GroundingDINO': 18}, 0.9920844327176781)

In [72]:
write_json_file(sft_data, output_path)
len(sft_data)

379

In [73]:
data2_turn_res = turn_into_sft_format(input_data2)
sft_data2, tool_statistic_dict2, average_tool_call2 = data2_turn_res["sft_data"], data2_turn_res["tool_statistic_dict"], data2_turn_res["average_tool_call"]
len(sft_data2), tool_statistic_dict2, average_tool_call2,

100%|██████████| 391/391 [00:01<00:00, 375.16it/s]


(347, {'GetBarInfo': 277, 'HighlightBox': 317, 'OCR': 2}, 1.7060518731988472)

In [74]:
write_json_file(sft_data2, output_path2)
len(sft_data2)

347